# Part III · Lesson 2 — Let a follow-up keep its meaning

## Import the document and model helpers

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

## Connect the two kinds of model

In [ ]:
load_dotenv()
# Groq generates answers; OpenAI creates search embeddings.
model = init_chat_model(
    "openai/gpt-oss-120b", model_provider="groq", reasoning_effort="low",
)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## Find the supplied text documents

In [ ]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Run this notebook with its own directory as the working directory.
data_dir = Path("../doc/smallStories")
text_files = sorted(data_dir.glob("*.txt"))
print("Files:", len(text_files))

## Keep text and source together

In [ ]:
documents = [
    Document(
        page_content=path.read_text(encoding="utf-8"),
        metadata={"source": path.name},
    )
    for path in text_files
]
print(documents[0].metadata)

## Split the documents into searchable pieces

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks = splitter.split_documents(documents)
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"{chunk.metadata['source']}:{index}"
print("Documents:", len(documents), "Chunks:", len(chunks))

## Build the vector index

In [ ]:
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

## Prepare readable context with source labels

In [ ]:
def format_context(documents: list[Document]) -> str:
    return "\n\n".join(
        f"[{doc.metadata['source']}]\n{doc.page_content}"
        for doc in documents
    )

## Keep the conversation as typed messages

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_history = []

## Describe the question-rewriting job

In [ ]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the latest question so it makes sense alone. "
     "Use the history to resolve references. Do not answer it. Return only the question."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

## Rewrite only when there is earlier history

In [ ]:
def contextualize_question(question: str) -> str:
    if not chat_history:
        return question
    prompt = rewrite_prompt.invoke({"history": chat_history, "question": question})
    return model.invoke(prompt).text.strip()

## Describe the answering job separately

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Use only the context for factual claims. History helps resolve references. "
     "If the context is insufficient, say you could not find the answer. "
     "Cite filenames in brackets.\nContext:\n{context}"),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

## Run and store one conversation turn

In [ ]:
def ask_conversational_rag(question: str) -> dict:
    standalone_question = contextualize_question(question)
    retrieved_documents = retriever.invoke(standalone_question)
    prompt = answer_prompt.invoke({
        "history": chat_history, "question": question,
        "context": format_context(retrieved_documents),
    })
    answer = model.invoke(prompt).text
    chat_history.extend([HumanMessage(question), AIMessage(answer)])
    return {
        "question": question, "standalone_question": standalone_question,
        "answer": answer, "documents": retrieved_documents,
    }

## Ask the first question

In [ ]:
first_turn = ask_conversational_rag("What happened to the grasshopper in winter?")
print("Search question:", first_turn["standalone_question"])
print("Answer:", first_turn["answer"])

## Ask a question that needs the previous turn

In [ ]:
second_turn = ask_conversational_rag("Who helped him?")
print("Original:", second_turn["question"])
print("Search question:", second_turn["standalone_question"])
print("Sources:", [doc.metadata["source"] for doc in second_turn["documents"]])
print("Answer:", second_turn["answer"])
print("History messages:", len(chat_history))